# Lesson 4 · Pandas in the Real World: Cleaning, Missing Values, Dates & Transformations
### Becoming a data detective: 80% of real data science is here

| | |
|---|---|
| **Type** | SOLUTIONS 04 |
| **Time** | about 150 minutes |
| **Level** | Beginner → Intermediate |
| **Prerequisites** | Lesson 3 (selecting, filtering, importing) |

**By the end of this notebook you will be able to:**

- Audit a dataset against the **seven data-quality dimensions**
- Find and remove duplicates; standardise messy text categories
- Detect impossible values, unit mix-ups and decimal-slip typos, and decide what to do with them
- Handle missing values with at least four strategies and *measure* which is best
- Parse dates, extract date parts, resample, roll and shift time series; handle time zones
- Transform columns with `map`, `apply`, `np.select`, `pd.cut` and package everything in a reusable `clean()` function

## 🛠️ Setup (identical to the lesson notebook)

In [1]:

import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check, first_look

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
raw = pd.read_csv(DATA_DIR / "market_prices_raw.csv")        # the MESSY file
clean = pd.read_csv(DATA_DIR / "market_prices_clean.csv")    # the tidy twin (our "answer key" for experiments)
print("Setup complete ✔  raw:", raw.shape, "| clean:", clean.shape)


Setup complete ✔  raw: (15093, 9) | clean: (15048, 9)


### 🧩 Problem 4.1: Rows left after removing duplicates

Remove exact duplicate rows from `raw`. How many rows remain?

*Expected type:* integer.

In [2]:
answer = len(raw.drop_duplicates())

check('04-p1', answer)

✅ 04-p1: correct! Nicely done.


True

### 🧩 Problem 4.2: How many distinct counties after cleaning?

Clean the `county` column of `raw` by: stripping spaces, replacing hyphens `-` with spaces, and applying **title case**. How many distinct counties are there afterwards?

*Expected type:* integer.

In [3]:
c = raw["county"].str.strip().str.replace("-", " ", regex=False).str.title()
answer = int(c.nunique())

check('04-p2', answer)

✅ 04-p2: correct! Nicely done.


True

### 🧩 Problem 4.3: Eldoret maize, in the right units

Starting from `raw` **after dropping duplicates by `record_id`**: rows with `price_unit == 'KES/90kg bag'` must be converted to KES per kg (divide the price by 90). Then compute the **median** `price_kes_per_kg` of **Maize** at **Eldoret Municipal Market** (NaNs are ignored automatically). Round to 2 decimals.

*Expected type:* float.

In [4]:
d = raw.drop_duplicates(subset="record_id").copy()
bag = d["price_unit"] == "KES/90kg bag"
d.loc[bag, "price_kes_per_kg"] = d.loc[bag, "price_kes_per_kg"] / 90
m = (d["crop"] == "Maize") & (d["market"] == "Eldoret Municipal Market")
answer = round(float(d.loc[m, "price_kes_per_kg"].median()), 2)

check('04-p3', answer)

✅ 04-p3: correct! Nicely done.


True

### 🧩 Problem 4.4: How much price data is missing?

After dropping duplicates by `record_id`, what **percentage** of `price_kes_per_kg` values are missing? Round to 2 decimals (e.g. `2.35` for 2.35%).

*Expected type:* float.

In [5]:
d = raw.drop_duplicates(subset="record_id")
answer = round(float(d["price_kes_per_kg"].isna().mean() * 100), 2)

check('04-p4', answer)

✅ 04-p4: correct! Nicely done.


True

### 🧩 Problem 4.5: Impossible values

After dropping duplicates by `record_id`, count the rows that have **a price ≤ 0 OR a negative volume** (a row meeting both conditions counts once).

*Expected type:* integer.

In [6]:
d = raw.drop_duplicates(subset="record_id")
answer = int(((d["price_kes_per_kg"] <= 0) | (d["volume_kg"] < 0)).sum())

check('04-p5', answer)

✅ 04-p5: correct! Nicely done.


True

### 🧩 Problem 4.6: Fill tomato gaps with a group median

From `raw` (after dropping duplicates by `record_id`), keep only **Tomatoes** rows. Fill missing `price_kes_per_kg` with the **median price of the same market**. (Ignore other issues such as typos.) Return the **mean** of the filled price column, rounded to 2 decimals.

*Expected type:* float.

In [7]:
d = raw.drop_duplicates(subset="record_id")
t = d[d["crop"] == "Tomatoes"].copy()
med = t.groupby("market")["price_kes_per_kg"].transform("median")
answer = round(float(t["price_kes_per_kg"].fillna(med).mean()), 2)

check('04-p6', answer)

✅ 04-p6: correct! Nicely done.


True

### 🧩 Problem 4.7: Records in Q4 2023

Using `raw` after dropping duplicates by `record_id`: parse `date` as datetime and count the records dated in **Q4 2023** (October–December 2023).

*Expected type:* integer.

In [8]:
d = raw.drop_duplicates(subset="record_id").copy()
d["date"] = pd.to_datetime(d["date"])
answer = int(((d["date"].dt.year == 2023) & (d["date"].dt.quarter == 4)).sum())

check('04-p7', answer)

✅ 04-p7: correct! Nicely done.


True

### 🧩 Problem 4.8: Peak month at Nakuru

Using the tidy `clean` table: parse `date`, select **Maize** at **Nakuru Municipal Market**, set the date as the index, and **resample by month-end** (`'ME'`) taking the mean price. Return the month with the highest average as a string like `'2023-02'`.

*Expected type:* string `YYYY-MM`.

In [9]:
c = clean.copy()
c["date"] = pd.to_datetime(c["date"])
s = c[(c["crop"] == "Maize") & (c["market"] == "Nakuru Municipal Market")].set_index("date")["price_kes_per_kg"]
answer = s.resample("ME").mean().idxmax().strftime("%Y-%m")

check('04-p8', answer)

✅ 04-p8: correct! Nicely done.


True

### 🧩 Problem 4.9: How many 'high' price records?

On `clean`, create price bands with `pd.cut`: **low** (0–50], **medium** (50–100], **high** (100–∞). How many rows fall in the **high** band?

*Expected type:* integer.

In [10]:
band = pd.cut(clean["price_kes_per_kg"], bins=[0, 50, 100, np.inf], labels=["low", "medium", "high"])
answer = int((band == "high").sum())

check('04-p9', answer)

✅ 04-p9: correct! Nicely done.


True

### ✍️ Problem 4.10: Your decision log

Two analysts disagree about the **decimal-slip prices** (e.g. a 60 KES/kg tomato recorded as 600). Analyst A wants to *drop those rows*; Analyst B wants to *divide by 10* to correct them. In 4–6 sentences, argue which approach you would choose **and** what you would write in the decision log. Consider: how sure are you of the cause? how many rows? what would a wrong correction cost?

**Model answer**

A strong answer weighs *certainty*: dividing by 10 is only safe if the slip is certain (a 10× jump in a series that is otherwise stable is *probably* a typo, but not provably). Dropping removes a whole record (including its volume, date, market) over one bad field. A middle path: **set the price to missing, impute it from neighbouring weeks, and flag `price_was_invalid=True`**: avoids inventing certainty, keeps the other fields, and lets later analysis exclude or test the repaired rows. Decision log: *'32 prices >3× or <⅓ of their crop-market median were set to NaN then interpolated; flagged in `price_was_invalid`. Cause suspected: decimal slip. Not corrected manually because cause unverified.'*